# Ch.4 — Parallelism & Distributed Training (Azure ML Supplement)

**No local GPU required.** This notebook shows how to run distributed training (PyTorch DDP + LoRA) on Azure ML compute clusters. You'll submit jobs to multi-node GPU clusters and monitor training remotely.

**What this notebook demonstrates:**
1. Azure ML workspace connection and authentication
2. Multi-node GPU cluster setup (2-8 nodes with A100 or V100 GPUs)
3. PyTorch DistributedDataParallel (DDP) configuration for Azure ML
4. LoRA fine-tuning job submission to Azure ML compute
5. Training job monitoring and cost tracking
6. Model artifact download from Azure ML
7. Cost comparison: Azure ML vs local RTX 4090

**Prerequisites:**
- Azure subscription with Azure ML workspace created
- Azure ML SDK v2 installed: `pip install azure-ai-ml azure-identity`
- (Optional) Azure CLI: `az login` for authentication

**WARNING:** Multi-node GPU training on Azure ML incurs costs ($2-10/hour per node depending on GPU type). Monitor your spending carefully.

## 1 · Azure ML Credentials Setup

**IMPORTANT SECURITY NOTE:** The pre-push Git hook will strip these credentials before any commit reaches the remote repository. However, you should still:
- Never commit credentials to personal repositories
- Use Azure Key Vault for production workloads
- Rotate keys regularly
- Use Managed Identity when running on Azure compute

**To get these values:**
1. Go to Azure Portal → Machine Learning → Your Workspace
2. Click "Overview" to find: Subscription ID, Resource Group, Workspace Name
3. For region: Check "Location" field (e.g., `eastus`, `westus2`, `westeurope`)

In [ ]:
# TODO: Implement this cell


## 2 · Azure ML SDK Setup

Install dependencies and import Azure ML SDK v2.

**Installation (run once in your environment):**
```bash
pip install azure-ai-ml azure-identity
```

In [ ]:
# TODO: Implement this cell


## 3 · Connect to Azure ML Workspace

Authenticate and connect to your Azure ML workspace.

**Authentication methods:**
1. **DefaultAzureCredential** (recommended): Tries multiple auth methods (CLI, Managed Identity, etc.)
2. **InteractiveBrowserCredential**: Opens browser for manual login

If you've run `az login` in your terminal, `DefaultAzureCredential` should work automatically.

In [ ]:
# TODO: Implement this cell


## 4 · Create Multi-Node GPU Compute Cluster

Azure ML compute clusters can scale from 0 to N nodes automatically. We'll create a cluster configured for distributed training.

**Cluster configuration:**
- **VM size:** `Standard_NC24ads_A100_v4` (1× A100 80GB per node) or `Standard_NC6s_v3` (1× V100 16GB, cheaper)
- **Min nodes:** 0 (scales down to zero when idle → no cost)
- **Max nodes:** 4 (can run 4-node distributed training)
- **Idle time:** 120 seconds (how long to wait before scaling down)

**Cost estimates (as of 2025):**
- `Standard_NC24ads_A100_v4` (A100 80GB): ~$3.67/hour per node
- `Standard_NC6s_v3` (V100 16GB): ~$0.90/hour per node
- `Standard_NC12s_v3` (2× V100 16GB): ~$1.80/hour per node

In [ ]:
# TODO: Implement this cell


## 5 · PyTorch DDP Training Script (Azure ML Compatible)

This cell creates the training script that will run on each node in the distributed cluster.

**Key differences from local DDP:**
1. Azure ML sets environment variables: `RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT`
2. Use `PyTorchDistribution` in job config to handle process initialization
3. Each node runs multiple processes (one per GPU)

**LoRA configuration:**
- Rank: 16 (sweet spot for quality vs memory)
- Target modules: attention layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`)
- Trainable params: ~0.5% of Llama-3-8B (42M / 8B)

In [ ]:
# TODO: Implement this cell


## 6 · Submit Distributed Training Job to Azure ML

Now we submit the training job to our GPU cluster. Azure ML will:
1. Spin up the requested number of nodes (scales from 0 → 2 nodes)
2. Install dependencies on each node
3. Run the training script with PyTorch DDP configuration
4. Collect logs and outputs
5. Scale down to 0 nodes after job completes (no idle cost)

**Job configuration:**
- **Instance count:** 2 nodes (each with 1 V100 GPU) → 2 GPUs total
- **Process count per node:** 1 (since Standard_NC6s_v3 has 1 GPU)
- **Distribution:** PyTorchDistribution (handles DDP process spawning)
- **Environment:** PyTorch 2.0 + Transformers + PEFT

In [ ]:
# TODO: Implement this cell


## 7 · Monitor Training Job

Poll the job status and display live updates. You can also monitor in Azure ML Studio (click the URL from previous cell).

**Job lifecycle:**
1. **Queued:** Waiting for compute nodes to spin up
2. **Preparing:** Installing environment (conda packages)
3. **Running:** Training in progress
4. **Completed:** Success! Artifacts available for download
5. **Failed:** Check logs in Azure ML Studio for error details

**Typical timeline:**
- Node provisioning: 3-5 minutes
- Environment setup: 5-10 minutes (first run; cached on subsequent runs)
- Training: 30-60 minutes (depends on dataset size and epochs)

In [ ]:
# TODO: Implement this cell


## 8 · Download Trained LoRA Adapters

After training completes, download the LoRA adapter weights from Azure ML.

**What you get:**
- `adapter_model.bin`: LoRA adapter weights (~84 MB for rank=16)
- `adapter_config.json`: LoRA configuration (rank, target modules, etc.)
- `tokenizer_config.json`, `tokenizer.json`: Tokenizer files

**Local inference:**
```python
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, "./downloaded-adapters")

# Run inference
outputs = model.generate(...)
```

In [ ]:
# TODO: Implement this cell


## 9 · Cost Comparison: Azure ML vs Local RTX 4090

Let's compare the economics of distributed training on Azure ML vs purchasing local hardware.

**Scenario:** Fine-tune Llama-3-8B with LoRA for 2 days

### Azure ML (2× Standard_NC6s_v3, V100 16GB)
- Hourly cost: $0.90/hour × 2 nodes = $1.80/hour
- Training time: 2 days (48 hours)
- **Total cost: $86.40** (one-time, no upfront investment)
- Pros: Pay-per-use, no maintenance, instant scaling, latest GPUs
- Cons: Slower than A100, data transfer costs, egress fees

### Azure ML (2× Standard_NC24ads_A100_v4, A100 80GB)
- Hourly cost: $3.67/hour × 2 nodes = $7.34/hour
- Training time: 1 day (24 hours, 2× faster than V100)
- **Total cost: $176.16** (one-time)
- Pros: Fastest training, 80GB VRAM (can do full fine-tuning), ZeRO-3 capable
- Cons: Higher hourly cost

### Local Hardware (2× RTX 4090, 24GB)
- Upfront cost: $1,600 × 2 = $3,200 (GPUs only; add $1,000+ for mobo/CPU/PSU/cooling)
- Training time: 2 days (48 hours)
- Electricity: ~900W × 48h × $0.12/kWh = $5.18
- **Total cost: $4,200 upfront + $5.18 per training run**
- Break-even: $4,200 / $86.40 = **49 training runs** (vs V100 on Azure)
- Pros: Unlimited runs after break-even, no data transfer, local control
- Cons: Upfront investment, maintenance, depreciation, single location

### Recommendation
- **Experimentation phase (1-10 runs):** Use Azure ML V100 ($86/run) — no upfront cost, fast iteration
- **Production phase (50+ runs/year):** Purchase RTX 4090s ($4,200 upfront) — break-even at 49 runs
- **Large models (70B+):** Use Azure ML A100 80GB ($176/run) — cannot fit on RTX 4090
- **Burst workloads:** Combine local RTX 4090 (baseline) + Azure ML (overflow) for cost optimization

In [ ]:
# TODO: Implement this cell


## 10 · Advanced: ZeRO-2 Distributed Training (4+ Nodes)

For larger models or full fine-tuning (not just LoRA), you can use **DeepSpeed ZeRO-2** to shard optimizer states across GPUs.

**When to use ZeRO-2:**
- Full fine-tuning (not LoRA): Need to update all 8B parameters
- Memory-constrained: 24GB VRAM not enough for full fine-tuning (104GB required)
- 4+ GPUs available: ZeRO-2 shards optimizer across GPUs (64GB ÷ 4 = 16GB per GPU)

**Memory breakdown with ZeRO-2 (4× V100 16GB):**
- Parameters (FP16, replicated): 16 GB per GPU
- Optimizer states (sharded): 16 GB per GPU (64GB total ÷ 4)
- Gradients (sharded): 4 GB per GPU (16GB total ÷ 4)
- Activations: 8 GB per GPU
- **Total: 44 GB per GPU** ❌ (still exceeds 16GB V100)

**Solution:** Use A100 40GB or enable gradient checkpointing (reduces activations 75%).

This cell shows how to configure DeepSpeed ZeRO-2 for Azure ML.

In [ ]:
# TODO: Implement this cell


## 11 · Cluster Cleanup

After completing your experiments, you can delete the compute cluster to ensure no accidental costs.

**Note:** Our cluster is configured with `min_instances=0`, so it scales down automatically when idle. You only pay when jobs are running. Deleting the cluster is optional — it's safe to leave it for future use.

**When to delete:**
- Finished with experiments for the quarter
- Need to free up subscription quota for other resources
- Want to recreate cluster with different VM size

**To delete:** Uncomment and run the cell below.

In [ ]:
# TODO: Implement this cell


## Summary & Next Steps

**What you learned:**
1. ✅ Azure ML workspace connection and authentication
2. ✅ Multi-node GPU cluster creation (auto-scaling 0 → N nodes)
3. ✅ PyTorch DistributedDataParallel (DDP) training on Azure ML
4. ✅ LoRA fine-tuning for memory-efficient distributed training
5. ✅ Job submission, monitoring, and artifact download
6. ✅ Cost comparison: Azure ML vs local hardware
7. ✅ DeepSpeed ZeRO-2 configuration for large-scale training

**Key takeaways:**
- **LoRA + DDP:** Train Llama-3-8B on 2× V100 16GB for $86 (2 days)
- **Break-even:** 49 training runs (Azure V100 vs local RTX 4090)
- **Scaling:** Azure ML auto-scales from 0 → N nodes → 0 (pay per use)
- **ZeRO-2:** Full fine-tuning requires A100 40GB+ (44GB per GPU with 4-node sharding)

**Next steps:**
1. **Ch.5 (Inference Optimization):** Deploy your fine-tuned LoRA adapters with vLLM
2. **Production:** Set up Azure ML pipelines for automated retraining
3. **Monitoring:** Integrate with Azure Monitor for cost tracking and alerting
4. **Experimentation:** Try ZeRO-3 (shard parameters) for 70B+ models

**Resources:**
- [Azure ML documentation](https://docs.microsoft.com/en-us/azure/machine-learning/)
- [PyTorch DDP tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)
- [PEFT (LoRA) library](https://github.com/huggingface/peft)
- [DeepSpeed ZeRO](https://www.deepspeed.ai/tutorials/zero/)